# CW-ODMR Demo: NV Center Spin Resonance Spectroscopy

This notebook walks through a complete **Continuous-Wave Optically Detected Magnetic Resonance (CW-ODMR)** measurement using a Red Pitaya as the controller.

## What is CW-ODMR?
The NV center in diamond has a ground-state spin resonance near **2.87 GHz** at zero magnetic field.  
When the applied microwave frequency matches this resonance, spin flips reduce the fluorescence — a dip in photodiode voltage.  
Sweeping frequency and recording voltage gives us the spectrum; its position encodes **magnetic field** and its width encodes **spin coherence**.

## What this notebook covers
| Section | Hardware needed? |
|---|---|
| 1. Imports & configuration | No |
| 2. Pre-flight register verification | No |
| 3. Hardware connection (SSH + PyRPL) | Yes |
| 4. ADF4355 initialisation | Yes |
| 5. Quick sanity-check sweep | Yes |
| 6. Full frequency sweep | Yes |
| 7. Spectrum analysis & fitting | No (uses saved data) |
| 8. Save results | No |

## 1. Imports & Configuration

In [ ]:
import os
import sys
import logging
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.optimize import curve_fit

# Add project root to path so imports work when running from notebooks/
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hardware.utils import verify_frequency, calc_registers, ADF4355Config, DEFAULT_CONFIG

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (10, 4)

print('Imports OK')

In [ ]:
# Logging — change to logging.DEBUG to see every socket command/response
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(name)-28s  %(levelname)s  %(message)s',
    datefmt='%H:%M:%S',
    force=True,
)
logger = logging.getLogger('odmr_demo')
logger.info('Logging ready')

In [ ]:
# ── Hardware connection ───────────────────────────────────────────────────────
HOSTNAME    = os.getenv('REDPITAYA_HOST', 'rp-XXXXXX.local')
PASSWORD    = os.getenv('REDPITAYA_PASSWORD', '')
ADC_CHANNEL = 1           # 1 = IN1, 2 = IN2

# ── Sweep parameters ─────────────────────────────────────────────────────────
F_START  = 2.70e9         # Hz
F_STOP   = 3.00e9         # Hz
N_POINTS = 301
DWELL_S  = 20e-3          # scope integration window per point (seconds)

# ── Output ───────────────────────────────────────────────────────────────────
DATA_DIR = PROJECT_ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)

eta_s = N_POINTS * (DWELL_S + 0.003)
print(f'Host      : {HOSTNAME}')
print(f'Sweep     : {F_START/1e9:.3f} – {F_STOP/1e9:.3f} GHz  |  {N_POINTS} points')
print(f'Dwell     : {DWELL_S*1e3:.0f} ms/point')
print(f'Est. time : {eta_s:.0f} s  (~{eta_s/60:.1f} min)')

## 2. Pre-flight: Offline Register Verification

Before touching hardware, verify that the target frequencies are achievable and inspect the ADF4355 register values.  
This step needs no network connection — failures here are purely arithmetic.

In [ ]:
# Inspect the NV zero-field frequency in detail
_ = verify_frequency(2.87e9)

In [ ]:
# Validate every planned sweep frequency — catches range errors before hardware is touched
sweep_freqs = np.linspace(F_START, F_STOP, N_POINTS)
errors_hz   = np.array([calc_registers(f)['_meta']['error_hz'] for f in sweep_freqs])

print(f'Sweep points : {N_POINTS}')
print(f'Max freq error : {errors_hz.max():.4f} Hz')
print(f'Mean freq error: {errors_hz.mean():.4f} Hz')
assert errors_hz.max() < 1.0, 'Some frequencies have >1 Hz error — check ADF4355Config'
print('✓ All sweep frequencies achievable with sub-Hz error')

In [ ]:
# Plot frequency error across the sweep
fig, ax = plt.subplots(figsize=(9, 2.5))
ax.plot(sweep_freqs / 1e9, errors_hz, color='steelblue', linewidth=0.8)
ax.set_xlabel('Frequency (GHz)')
ax.set_ylabel('Freq error (Hz)')
ax.set_title('ADF4355 frequency quantisation error across sweep')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Hardware Connection

Two independent connections to the Red Pitaya:

| Connection | Transport | Purpose |
|---|---|---|
| `RPSynthTransport` | Paramiko SSH → TCP socket | ADF4355 SPI (bit-bangs on RP ARM) |
| `pyrpl.Pyrpl` | PyRPL TCP | Scope / ADC readout |

`connect()` uploads `rp_daemon.py` to `/tmp/` via SFTP and starts it in the background.  
If the daemon log shows an error later, SSH in and inspect `/tmp/adf4355_daemon.log`.

In [ ]:
from hardware import ADF4355, RPSynthTransport

if 'XXXXXX' in HOSTNAME:
    raise RuntimeError(
        'HOSTNAME is still a placeholder.\n'
        'Set the REDPITAYA_HOST environment variable before running this cell.'
    )

transport = RPSynthTransport(HOSTNAME, password=PASSWORD)
transport.connect()
logger.info('SPI transport connected')

In [ ]:
# Connect PyRPL scope for ADC readout (separate TCP connection)
try:
    from pyrpl import Pyrpl  # type: ignore[import-untyped]
    rp = Pyrpl(
        hostname=HOSTNAME,
        password=PASSWORD,
        gui=False,
        config=os.getenv('REDPITAYA_CONFIG', 'pyrpl_config.yml'),
    )
    logger.info('PyRPL scope connected')
except ImportError:
    logger.error('pyrpl is not installed — ADC readout will not work.')
    rp = None
except Exception as exc:
    logger.error('PyRPL connection failed: %s', exc)
    rp = None

## 4. ADF4355 Initialisation

The full power-up sequence writes all 13 registers in the order required by the datasheet (12 → 1, wait, 0).  
Log output shows the achieved frequency and divider values.

In [ ]:
synth = ADF4355(transport)
synth.init(f_init_hz=F_START)
logger.info('ADF4355 ready at %.3f GHz', F_START / 1e9)

In [ ]:
# Spot-check: tune to NV zero-field and back to sweep start
logger.info('Tuning to NV zero-field (2.87 GHz) …')
synth.set_frequency(2.87e9)

logger.info('Returning to sweep start (%.3f GHz) …', F_START / 1e9)
synth.set_frequency(F_START)

logger.info('Spot-check complete')

## 5. Sanity Check: 5-Point Mini-Sweep

Before committing to the full sweep, run 5 points to confirm the photodiode is responding and the ADC is reading sensible voltages.

In [ ]:
from experiments.odmr import ODMRSweep

test_sweep = ODMRSweep(
    synth, rp=rp, adc_channel=ADC_CHANNEL,
    f_start=2.82e9, f_stop=2.92e9, n_points=5, dwell_s=DWELL_S,
)

test_freqs, test_voltages = test_sweep.run()

print('Frequency (GHz)  │  Voltage (mV)')
print('─' * 33)
for f, v in zip(test_freqs, test_voltages):
    print(f'  {f/1e9:.4f}         │  {v*1e3:7.2f}')

In [ ]:
# Interpret the sanity check
v_mean = test_voltages.mean()
v_std  = test_voltages.std()
v_cv   = v_std / v_mean * 100  # coefficient of variation

print(f'Mean   : {v_mean*1e3:.2f} mV')
print(f'Std    : {v_std*1e3:.2f} mV  (CV = {v_cv:.1f}%)')

if v_mean < 1e-3:
    logger.warning('Mean voltage is very low (<1 mV) — check photodiode connection and laser.')
elif v_cv > 30:
    logger.warning('High variance (CV %.0f%%) — check for laser or vibration instability.', v_cv)
else:
    logger.info('Signal looks healthy  ✓')

## 6. Full ODMR Sweep

Each frequency step:
1. Sends a `FREQ` command over the TCP socket (~1 ms round-trip)
2. Daemon runs the 8-step ADF4355 sequence and waits 2 ms for PLL lock
3. Scope acquires `DWELL_S` seconds of photodiode signal and returns the mean

The iterator pattern is used here so we can print progress every 30 points.

In [ ]:
sweep = ODMRSweep(
    synth, rp=rp, adc_channel=ADC_CHANNEL,
    f_start=F_START, f_stop=F_STOP, n_points=N_POINTS, dwell_s=DWELL_S,
)
print(f'Ready: {N_POINTS} pts × ({DWELL_S*1e3:.0f} ms dwell + ~2 ms lock)  ≈ {N_POINTS*(DWELL_S+0.003):.0f} s')

In [ ]:
# Run the sweep with progress logging every 30 points
t0 = time.monotonic()

try:
    for i, freq in enumerate(sweep):
        v = sweep.read_voltage()
        sweep.record(v)

        if (i + 1) % 30 == 0 or i == 0 or i == N_POINTS - 1:
            elapsed   = time.monotonic() - t0
            remaining = elapsed / (i + 1) * (N_POINTS - i - 1)
            logger.info(
                '%3d/%d   %.4f GHz   V = %.2f mV   [%ds, ~%ds left]',
                i + 1, N_POINTS, freq / 1e9, v * 1e3, int(elapsed), int(remaining),
            )

except KeyboardInterrupt:
    logger.warning('Interrupted at point %d — saving partial data', i + 1)

freqs, voltages = sweep.result()
logger.info('Sweep done: %d points in %.1f s', len(freqs), time.monotonic() - t0)

## 7. Spectrum Analysis

### 7a. Raw spectrum

In [ ]:
fig, ax = plt.subplots()
ax.plot(freqs / 1e9, voltages * 1e3, color='steelblue', linewidth=0.8, label='Raw')
ax.set_xlabel('Frequency (GHz)')
ax.set_ylabel('Photodiode signal (mV)')
ax.set_title('CW-ODMR Raw Spectrum')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7b. Smoothing and dip detection

A Savitzky–Golay filter preserves the dip shape while reducing point-to-point noise.  
We then pick the minimum as an initial guess for the resonance position.

In [ ]:
voltages_smooth = savgol_filter(voltages, window_length=11, polyorder=3)

i_dip     = int(np.argmin(voltages_smooth))
f_dip     = freqs[i_dip]
v_dip     = voltages_smooth[i_dip]
baseline  = np.percentile(voltages_smooth, 90)   # high-signal baseline
contrast  = (baseline - v_dip) / baseline * 100

print(f'Dip position : {f_dip / 1e9:.6f} GHz')
print(f'Dip voltage  : {v_dip * 1e3:.2f} mV')
print(f'Baseline     : {baseline * 1e3:.2f} mV')
print(f'Contrast     : {contrast:.1f} %')

### 7c. Lorentzian fit

The NV ODMR resonance is well approximated by a Lorentzian dip:

$$V(f) = V_0 - A \frac{(\Gamma/2)^2}{(f - f_0)^2 + (\Gamma/2)^2}$$

where $f_0$ is the resonance frequency and $\Gamma$ is the FWHM linewidth.

In [ ]:
def lorentzian_dip(f, f0, gamma, A, V0):
    """Single Lorentzian dip — A and V0 in the same units as the data."""
    return V0 - A * (gamma / 2)**2 / ((f - f0)**2 + (gamma / 2)**2)

# Restrict fit to ±60 MHz around the detected dip to exclude the other resonance
FIT_WINDOW = 60e6
mask = np.abs(freqs - f_dip) < FIT_WINDOW

p0 = [f_dip, 10e6, baseline - v_dip, baseline]

fit_ok = False
try:
    popt, pcov = curve_fit(lorentzian_dip, freqs[mask], voltages[mask], p0=p0, maxfev=8000)
    perr = np.sqrt(np.diag(pcov))
    f0_fit, gamma_fit, A_fit, V0_fit = popt
    fit_ok = True

    print(f'Resonance  : {f0_fit/1e9:.6f} GHz  (±{perr[0]/1e6:.3f} MHz)')
    print(f'Linewidth  : {gamma_fit/1e6:.2f} MHz FWHM  (±{perr[1]/1e6:.2f} MHz)')
    print(f'Contrast   : {A_fit/V0_fit*100:.2f} %')

except RuntimeError as exc:
    logger.warning('Lorentzian fit did not converge: %s', exc)
    print('Fit failed — try adjusting FIT_WINDOW or initial guesses.')

In [ ]:
# Final annotated plot
f_fine = np.linspace(F_START, F_STOP, 2000)

fig, ax = plt.subplots()
ax.plot(freqs / 1e9, voltages * 1e3,
        color='steelblue', linewidth=0.7, alpha=0.5, label='Data')
ax.plot(freqs / 1e9, voltages_smooth * 1e3,
        color='steelblue', linewidth=1.4, label='Smoothed')

if fit_ok:
    ax.plot(f_fine / 1e9, lorentzian_dip(f_fine, *popt) * 1e3,
            color='tomato', linewidth=1.8,
            label=f'Fit  $f_0$ = {f0_fit/1e9:.4f} GHz,  $\\Gamma$ = {gamma_fit/1e6:.1f} MHz')
    ax.axvline(f0_fit / 1e9, color='tomato', linestyle='--', alpha=0.4, linewidth=1)

ax.axvline(2.87, color='grey', linestyle=':', alpha=0.6, linewidth=1, label='2.87 GHz (zero field)')

ax.set_xlabel('Frequency (GHz)', fontsize=12)
ax.set_ylabel('Photodiode signal (mV)', fontsize=12)
ax.set_title('CW-ODMR Spectrum — NV Center in Diamond', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Save Results

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
npz_path  = DATA_DIR / f'odmr_{timestamp}.npz'
png_path  = DATA_DIR / f'odmr_{timestamp}.png'

save_data = dict(
    freqs=freqs,
    voltages=voltages,
    f_start=F_START, f_stop=F_STOP,
    n_points=N_POINTS, dwell_s=DWELL_S,
    hostname=HOSTNAME,
)
if fit_ok:
    save_data.update(
        f0_fit=f0_fit,
        gamma_fit=gamma_fit,
        contrast_fit=A_fit / V0_fit,
    )

np.savez(npz_path, **save_data)
fig.savefig(png_path, dpi=150, bbox_inches='tight')

print(f'Data   saved → {npz_path}')
print(f'Figure saved → {png_path}')

In [ ]:
# Demonstrate reloading saved data (useful for offline analysis later)
loaded   = np.load(npz_path, allow_pickle=False)
print('Keys in saved file:', list(loaded.keys()))
print(f'Reloaded {len(loaded["freqs"])} frequency points')

## 9. Close Connections

The daemon process stays running on the RP — calling `transport.close()` only disconnects the socket and SSH session.  
A subsequent `transport.connect()` will upload a fresh daemon and reconnect.

In [ ]:
transport.close()
logger.info('All connections closed')